In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("../data")
amazon_books_ratings = pd.read_csv(f"{BASE_DIR}/amazon_books_reviews/Books_rating.csv")

In [4]:
amazon_books_ratings.head()

,Id,Title,User_id,review/helpfulness,review/score,review/time,review/summary
0,1882931173,Its Only Art If Its Well Hung!,AVCGYZL8FQQTD,7/7,4.0,940636800,Nice collection of Julie Strain images
1,0826414346,Dr. Seuss: American Icon,A30TK6U7DNS82R,10/10,5.0,1095724800,Really Enjoyed It
2,0826414346,Dr. Seuss: American Icon,A3UH4UZ4RSVO82,10/11,5.0,1078790400,Essential for every personal and Public Library
3,0826414346,Dr. Seuss: American Icon,A2MVUWT453QH61,7/7,4.0,1090713600,Phlip Nel gives silly Seuss a serious treatment
4,0826414346,Dr. Seuss: American Icon,A22X4XUPKF66MR,3/3,4.0,1107993600,Good academic overview


### Select columns

In [3]:
amazon_books_ratings = amazon_books_ratings[['Id', 'Title', 'User_id', 'review/helpfulness', 'review/score', 'review/time', 'review/summary']]

In [5]:
amazon_books_ratings.isna().sum()

Id                         0
Title                    208
User_id               561787
review/helpfulness         0
review/score               0
review/time                0
review/summary           407
dtype: int64

### remove null value

In [7]:
ratings = amazon_books_ratings.dropna(subset=["User_id"]).copy()

In [9]:
ratings.duplicated(subset=["User_id", "Id"]).sum()

np.int64(40599)

In [11]:
ratings = (
    ratings
    .sort_values("review/score", ascending=False)
    .drop_duplicates(subset=["User_id", "Id"], keep="first")
    .reset_index(drop=True)
)
ratings.duplicated(subset=["User_id", "Id"]).sum()

np.int64(0)

### rename columns

In [14]:
ratings = ratings.rename(columns={
    'Id': 'isbn',
    'Title': 'title',
    'User_id': 'user_id',
    'review/score': 'score',
    'review/time': 'review_time',
    'review/summary': 'review_summary',
    'review/helpfulness': 'helpfulness'
})

### convert timestemp

In [16]:
ratings['review_time'] = pd.to_datetime(ratings['review_time'], unit='s')

### Clean rating

In [18]:
ratings['score'] = pd.to_numeric(ratings['score'], errors='coerce')
ratings = ratings[(ratings['score'] >= 0) & (ratings['score'] <= 5)]

### Clean helpfulness

In [20]:
ratings[['helpful_yes', 'helpful_total']] = ratings['helpfulness'].str.split('/', expand=True)

ratings['helpful_yes'] = pd.to_numeric(ratings['helpful_yes'], errors='coerce')
ratings['helpful_total'] = pd.to_numeric(ratings['helpful_total'], errors='coerce')

In [25]:
ratings.drop(columns=['helpfulness'], inplace=True)

### Clean text columns

In [22]:
ratings['title'] = ratings['title'].str.strip()
ratings['review_summary'] = ratings['review_summary'].str.strip()

### Handle missing values

In [23]:
ratings = ratings.replace(['None', 'nan', ''], None)

In [26]:
ratings.head(20)

,isbn,title,user_id,score,review_time,review_summary,helpful_yes,helpful_total
0,0826414346,Dr. Seuss: American Icon,A14OJS0VWMOSWO,5.0,2004-11-11,A memorably excellent survey of Dr. Seuss' man...,3,4
1,B000NSLVCU,The Idea of History,A18SQGYBKS852K,5.0,2006-11-09,"Yes, it is cheaper than the University Bookstore",1,11
2,0826414346,Dr. Seuss: American Icon,A2RSSXTDZDUSH4,5.0,2009-01-06,Academia At It's Best,0,0
3,0826414346,Dr. Seuss: American Icon,A30TK6U7DNS82R,5.0,2004-09-21,Really Enjoyed It,10,10
4,0826414346,Dr. Seuss: American Icon,A3UH4UZ4RSVO82,5.0,2004-03-09,Essential for every personal and Public Library,10,11
5,0826414346,Dr. Seuss: American Icon,A25MD5I2GUIW6W,5.0,2008-05-04,And to think that I read it on the tram!,0,0
6,B000NSGW7E,Ghost Story,A9Z9ETUI48IVC,5.0,2009-11-15,Shivers me timbers!!!!!,0,4
7,B000NSGW7E,Ghost Story,APW0VLW2NWO98,5.0,2009-04-29,Great Transactions!,0,4
8,B000NSGW7E,Ghost Story,AD74ICXKB2O0B,5.0,2010-02-06,unbelievable condition,0,5
9,B000NSLVCU,The Idea of History,AI1QNMVF2E3TN,5.0,2003-07-01,R. G. Collingwood's Most Famous Book,28,29


In [27]:
# save output
ratings.to_csv(BASE_DIR / "clean" / "amazon_books_ratings_clean_2.csv", index=False)